<a href="https://colab.research.google.com/github/SebastianoColombini/NN_DL_project/blob/main/prova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Import section
from google.colab import drive
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms


In [9]:
version = {
    "version_name": "alpha",
    "lr": None
}

## SALICON Dataset Loading

To use the SALICON dataset, we need to define a custom PyTorch `Dataset` class that handles loading the images and their corresponding saliency maps. This example assumes you have your dataset organized with separate directories for training images and saliency maps.

**Note:** You will need to adjust `root_dir_images`, `root_dir_saliency_maps`, and the image/saliency map loading logic to match your specific dataset structure and file formats.

In [ ]:
class SALICONDataset(Dataset):
    def __init__(self, image_root_dir, saliency_map_root_dir, transform=None):
        self.image_root_dir = image_root_dir
        self.saliency_map_root_dir = saliency_map_root_dir
        self.transform = transform

        # Assuming image and saliency map filenames correspond
        self.image_filenames = sorted(os.listdir(image_root_dir))
        self.saliency_map_filenames = sorted(os.listdir(saliency_map_root_dir))

        # Basic check to ensure same number of images and maps
        assert len(self.image_filenames) == len(self.saliency_map_filenames), \
            "Number of images and saliency maps must be the same"

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = os.path.join(self.image_root_dir, self.image_filenames[idx])
        saliency_name = os.path.join(self.saliency_map_root_dir, self.saliency_map_filenames[idx])

        image = Image.open(img_name).convert('RGB')
        saliency_map = Image.open(saliency_name).convert('L') # Convert to grayscale for saliency

        if self.transform:
            image = self.transform(image)
            saliency_map = self.transform(saliency_map)

        return image, saliency_map

LOCAL_DIR = "/DB"
os.makedirs(LOCAL_DIR, exist_ok=True)

transforms_train = transforms.Compose([
    transforms.Resize((480, 640)), # Resize images to a common size
    transforms.ToTensor(),        # Convert PIL Image to PyTorch Tensor
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize for pre-trained models, if applicable
])


TRAIN_IMAGES_DIR = os.path.join(LOCAL_DIR, 'SALICON', 'train', 'images')
TRAIN_SALIENCY_MAPS_DIR = os.path.join(LOCAL_DIR, 'SALICON', 'train', 'saliency_maps')

# Create the dataset and dataloader
salicon_dataset = SALICONDataset(
    image_root_dir=TRAIN_IMAGES_DIR,
    saliency_map_root_dir=TRAIN_SALIENCY_MAPS_DIR,
    transform=transforms_train
)

print(f"Number of samples in SALICON training dataset: {len(salicon_dataset)}")
print(f"Number of batches in DataLoader: {len(dataloader)}")



In [10]:
# IMPORT DI GOOGLE DRIVE
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/NN_DL_Proj"
VERSION_DIR = os.path.join(OUTPUT_DIR, version["version_name"])
SNAPSHOTS_DIR = os.path.join(VERSION_DIR, "snapshots")


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VERSION_DIR, exist_ok=True)
os.makedirs(SNAPSHOTS_DIR, exist_ok=True)

# Save the version JSON to a file in VERSION_DIR
version_file_path = os.path.join(VERSION_DIR, "version.json")
with open(version_file_path, 'w') as f:
    json.dump(version, f, indent=4)

Mounted at /content/drive


In [11]:
def normalize_map(s_map):
    """Normalizza la mappa affinché la somma di tutti i pixel sia pari a 1 (Distribuzione di probabilità)"""
    batch_size = s_map.size(0)
    s_map_flat = s_map.view(batch_size, -1)
    # Evitiamo la divisione per zero aggiungendo un piccolo epsilon
    s_map_sum = torch.sum(s_map_flat, dim=1, keepdim=True) + 1e-7
    s_map_norm = s_map_flat / s_map_sum
    return s_map_norm.view_as(s_map)

def kld_loss(pred, gt):
    """Kullback-Leibler Divergence Loss"""
    pred = normalize_map(pred)
    gt = normalize_map(gt)
    eps = 1e-7
    # Formula KLD: Somma( GT * log(GT / (Pred + eps)) )
    loss = gt * torch.log(gt / (pred + eps) + eps)
    return torch.mean(torch.sum(loss.view(loss.size(0), -1), dim=1))

def cc_loss(pred, gt):
    """Pearson Correlation Coefficient Loss (espressa come 1 - CC per minimizzarla)"""
    pred_flat = pred.view(pred.size(0), -1)
    gt_flat = gt.view(gt.size(0), -1)

    # Calcolo delle medie
    pred_mean = torch.mean(pred_flat, dim=1, keepdim=True)
    gt_mean = torch.mean(gt_flat, dim=1, keepdim=True)

    # Centratura rispetto alla media
    pred_center = pred_flat - pred_mean
    gt_center = gt_flat - gt_mean

    # Formula di Pearson
    cc_num = torch.sum(pred_center * gt_center, dim=1)
    cc_den = torch.sqrt(torch.sum(pred_center ** 2, dim=1) * torch.sum(gt_center ** 2, dim=1) + 1e-7)

    cc = cc_num / cc_den
    return torch.mean(1 - cc) # Vogliamo massimizzare CC, quindi minimizziamo (1 - CC)

def sim_loss(pred, gt):
    """Histogram Similarity Loss (espressa come 1 - SIM)"""
    pred = normalize_map(pred)
    gt = normalize_map(gt)

    pred_flat = pred.view(pred.size(0), -1)
    gt_flat = gt.view(gt.size(0), -1)

    # Prende il minimo pixel per pixel tra le due distribuzioni normalizzate
    sim = torch.sum(torch.min(pred_flat, gt_flat), dim=1)
    return torch.mean(1 - sim) # Vogliamo massimizzare la somiglianza

def single_loss(pred, gt, w_kld=1.0, w_cc=1.0, w_sim=1.0):
    """Combina le 3 perdite per un singolo ramo"""
    return w_kld * kld_loss(pred, gt) + w_cc * cc_loss(pred, gt) + w_sim * sim_loss(pred, gt)

def total_loss(final_map, s1, s2, s3, gt):
    """Calcola la loss totale applicando la Deep Supervision su tutti i rami"""
    l_final = single_loss(final_map, gt)
    l_s1 = single_loss(s1, gt)
    l_s2 = single_loss(s2, gt)
    l_s3 = single_loss(s3, gt)

    # Somma di tutte le loss (con pesi impostati a 1 come da impostazione standard)
    return l_final + l_s1 + l_s2 + l_s3

In [20]:
from torch.nn.modules import conv
# ENCODER
class Encoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.kernel_size=3
    self.stride=1
    self.padding=1

    self.encoder_cnn1 = nn.Sequential(
        nn.Conv2d(3, 64, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(64, 64, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
    )
    self.encoder_cnn2 = nn.Sequential(
        nn.Conv2d(64, 128, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(128, 128, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
    )
    self.encoder_cnn3 = nn.Sequential(
        nn.Conv2d(128, 256, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(256, 256, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(256, 256, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
    )
    self.encoder_cnn4 = nn.Sequential(
        nn.Conv2d(256, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(512, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(512, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.MaxPool2d(kernel_size=2,stride=2,padding=0)
    )
    self.encoder_cnn5 = nn.Sequential(
        nn.Conv2d(512, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(512, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True),
        nn.Conv2d(512, 512, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding),
        nn.ReLU(True)
    )

  def forward(self, x: torch.Tensor):
    out = []
    x = self.encoder_cnn1(x)
    x = self.encoder_cnn2(x)
    x = self.encoder_cnn3(x)
    out.append(x.copy())
    x = self.encoder_cnn4(x)
    out.append(x.copy())
    x = self.encoder_cnn5(x)
    out.append(x.copy())
    return out


In [24]:
# DECODER 3
class Decoder3(nn.Module):
  def __init__(self):
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.Conv2d(256, 1, kernel_size=1, stride=1, padding=0),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.Sigmoid()
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)

In [25]:
# DECODER 4
class Decoder4(nn.Module):
  def __init__(self):
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.Conv2d(512, 1, kernel_size=1, stride=1, padding=0),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.Sigmoid()
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)

In [26]:
# DECODER 5
class Decoder5(nn.Module):
  def __init__(self):
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.Conv2d(512, 1, kernel_size=1, stride=1, padding=0),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.ReLU(True),
        nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1),
        nn.Sigmoid()
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)

In [27]:
# FINAL LAYER
class FinalCov(nn.Module):
  def __init__(self):
    super().__init__()
    self.decoder_cnn = nn.Sequential(
        nn.ConvTranspose2d(3, 1, kernel_size=1, stride=1, padding=0),
        nn.Sigmoid()
    )

  def forward(self, x: torch.Tensor):
    return self.decoder_cnn(x)


In [28]:
# MAIN PHASE OF TRAINING
# Istanziazione dei componenti e aggancio all'ottimizzatore

batch_size = 16
dataloader = DataLoader(salicon_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

encoder = Encoder()
decoder3 = Decoder3()
decoder4 = Decoder4()
decoder5 = Decoder5()
final = FinalCov()


loss_fn = nn.MSELoss()
optimizer = optim.Adam(
    list(encoder.parameters()) +
    list(decoder3.parameters()) +
    list(decoder4.parameters()) +
    list(decoder5.parameters()) +
    list(final.parameters())
    , lr=0.001
)

# Rilevamento hardware (CUDA / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)
decoder3.to(device)
decoder4.to(device)
decoder5.to(device)
final.to(device)

# Loop delle epoche
num_epochs = 20
for epoch in range(num_epochs):

    # Attivazione dei moduli in modalità addestramento
    encoder.train()
    decoder3.train()
    decoder4.train()
    decoder5.train()
    final.train()

    for image_batch, _ in dataloader: # Le label di classificazione vengono ignorate

        # Allocazione del batch sul dispositivo di calcolo
        time = time.time()
        image_batch = image_batch.to(device)

        # Passaggio in avanti (Forward Pass)
        latent_features = encoder(image_batch)
        saliency_3 = decoder3(latent_features[0])
        saliency_4 = decoder4(latent_features[1])
        saliency_5 = decoder5(latent_features[2])
        fused_input=torch.cat((saliency_3, saliency_4, saliency_5),dim =1)
        reconstructed_images = final(fused_input)

        loss = total_loss(reconstructed_images, saliency_3, saliency_4, saliency_5, image_batch)

        optimizer.zero_grad()  # Svuotamento dei gradienti accumulati nei tensori
        loss.backward()       # Calcolo dei nuovi gradienti tramite backpropagation
        optimizer.step()       # Aggiornamento dei parametri della rete
        print(f"Epochs duration:{time - time.time()}")

NameError: name 'dataloader' is not defined